# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/anany/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-18 21:05:59.881613: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 21:05:59.887766: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-18 21:05:59.985520: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-18 21:05:59.985619: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-18 21:05:59.985685: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df.drop(columns=["Class"]).values.astype(np.float32)
y = df["Class"].values.astype(np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique class labels:", np.unique(y))


X shape: (178, 13)
y shape: (178,)
Unique class labels: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Training samples: 124
Test samples: 54


In [11]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

In [12]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [13]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_base = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 2s 61ms/step - loss: 1.0504 - accuracy: 0.4545 - val_loss: 0.9264 - val_accuracy: 0.7200
Epoch 2/20
13/13 [==============================] - 0s 17ms/step - loss: 0.7904 - accuracy: 0.8081 - val_loss: 0.6961 - val_accuracy: 0.8800
Epoch 3/20
13/13 [==============================] - 0s 13ms/step - loss: 0.5946 - accuracy: 0.8889 - val_loss: 0.5256 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 17ms/step - loss: 0.4417 - accuracy: 0.9495 - val_loss: 0.3867 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 18ms/step - loss: 0.3155 - accuracy: 0.9899 - val_loss: 0.2830 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 14ms/step - loss: 0.2243 - accuracy: 0.9798 - val_loss: 0.2085 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 26ms/step - loss: 0.1606 - accuracy: 0.9899 - val_loss: 0.1617 - val_accuracy: 0.9600
Epoch 8/20
13

In [18]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Base model test accuracy: {test_accuracy:.4f}")
print(f"Base model test loss: {test_loss:.4f}")

y_pred_base = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification Report - Base Model")
print(classification_report(y_true, y_pred_base, target_names=wine.target_names))


print("Confusion Matrix - Base Model")
print(confusion_matrix(y_true, y_pred_base))

Base model test accuracy: 1.0000
Base model test loss: 0.0452

Classification Report - Base Model
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix - Base Model
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


In [20]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

import os

def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_base = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model_base)

print(f"Base TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /tmp/tmpv8xsbkt3/assets


INFO:tensorflow:Assets written to: /tmp/tmpv8xsbkt3/assets


Base TFLite model size: 14.07 KB


2026-05-18 21:19:07.382033: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:19:07.382196: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:19:07.382885: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpv8xsbkt3
2026-05-18 21:19:07.385868: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:19:07.385988: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpv8xsbkt3
2026-05-18 21:19:07.391666: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 21:19:07.446823: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpv8xsbkt3
2026-05-18 21:19:07.465484: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 82603 m

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [21]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_index = input_details["index"]
    output_index = output_details["index"]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    y_pred = []

    for sample in X_test.astype(np.float32):
        sample = sample.reshape(1, -1)

        if input_details["dtype"] in (np.int8, np.uint8):
            sample_input = sample / input_scale + input_zero_point
            sample_input = np.round(sample_input)
            info = np.iinfo(input_details["dtype"])
            sample_input = np.clip(sample_input, info.min, info.max).astype(input_details["dtype"])
        else:
            sample_input = sample.astype(input_details["dtype"])

        interpreter.set_tensor(input_index, sample_input)
        interpreter.invoke()

        output = interpreter.get_tensor(output_index)

        if output_details["dtype"] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    accuracy = np.mean(y_pred == y_true)
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"{quant_type.upper()} TFLite accuracy: {accuracy:.4f}")

    print(f"\nClassification Report - {quant_type.upper()} TFLite")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))

    print(f"Confusion Matrix - {quant_type.upper()} TFLite")
    print(confusion_matrix(y_true, y_pred))

    return {
        "quant_type": quant_type,
        "filename": filename,
        "size_kb": file_size_kb(filename),
        "accuracy": accuracy,
        "y_pred": y_pred
    }

In [22]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quant_results = {}

quant_results["int8"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="int8",
    filename="model_int8.tflite"
)

quant_results["float16"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="float16",
    filename="model_float16.tflite"
)

quant_results["dynamic"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    quant_type="dynamic",
    filename="model_dynamic.tflite"
)

INFO:tensorflow:Assets written to: /tmp/tmp_ngdav8b/assets


INFO:tensorflow:Assets written to: /tmp/tmp_ngdav8b/assets
/home/anany/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-18 21:21:15.091356: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:21:15.091537: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:21:15.092094: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_ngdav8b
2026-05-18 21:21:15.094367: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:21:15.094406: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp_ngdav8b
2026-05-18 21:21:15.100072: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202


INT8 TFLite model size: 5.74 KB
INT8 TFLite accuracy: 1.0000

Classification Report - INT8 TFLite
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix - INT8 TFLite
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpn1hhnewd/assets


INFO:tensorflow:Assets written to: /tmp/tmpn1hhnewd/assets
2026-05-18 21:21:16.379755: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:21:16.379882: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:21:16.380211: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpn1hhnewd
2026-05-18 21:21:16.381424: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:21:16.381467: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpn1hhnewd
2026-05-18 21:21:16.385673: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 21:21:16.446173: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpn1hhnewd
2026-05-18 21:21:16.459358: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.95 KB
FLOAT16 TFLite accuracy: 1.0000

Classification Report - FLOAT16 TFLite
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix - FLOAT16 TFLite
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmps4rjvua7/assets


INFO:tensorflow:Assets written to: /tmp/tmps4rjvua7/assets



DYNAMIC TFLite model size: 8.17 KB
DYNAMIC TFLite accuracy: 1.0000

Classification Report - DYNAMIC TFLite
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix - DYNAMIC TFLite
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


2026-05-18 21:21:17.955708: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:21:17.955825: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:21:17.956098: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmps4rjvua7
2026-05-18 21:21:17.957635: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:21:17.957658: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmps4rjvua7
2026-05-18 21:21:17.961928: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 21:21:18.012515: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmps4rjvua7
2026-05-18 21:21:18.028705: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 72606 m

## Problem 1 - Part (c)

### Pruning

In [23]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

pruning_epochs = 10
pruning_batch_size = 8

steps_per_epoch = int(np.ceil(X_train_scaled.shape[0] / pruning_batch_size))
end_step = steps_per_epoch * pruning_epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.50,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step
)

print("Steps per epoch:", steps_per_epoch)
print("Pruning end step:", end_step)

Steps per epoch: 16
Pruning end step: 160


In [24]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

tf.random.set_seed(42)
np.random.seed(42)

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [25]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_pruned = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 4s 37ms/step - loss: 1.0966 - accuracy: 0.4444 - val_loss: 0.8750 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 16ms/step - loss: 0.8188 - accuracy: 0.7677 - val_loss: 0.6401 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 13ms/step - loss: 0.6039 - accuracy: 0.9192 - val_loss: 0.4661 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 17ms/step - loss: 0.4264 - accuracy: 0.9697 - val_loss: 0.3247 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 17ms/step - loss: 0.2884 - accuracy: 0.9899 - val_loss: 0.2285 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 14ms/step - loss: 0.1928 - accuracy: 0.9899 - val_loss: 0.1711 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 19ms/step - loss: 0.1375 - accuracy: 0.9899 - val_loss: 0.1310 - val_accuracy: 0.9600
Epoch 8/10
13

In [28]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

stripped_pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_model_pruned = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_model_pruned)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpi9trpzdm/assets


INFO:tensorflow:Assets written to: /tmp/tmpi9trpzdm/assets


Pruned TFLite model size: 14.14 KB


2026-05-18 21:28:59.727421: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:28:59.727531: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:28:59.727878: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpi9trpzdm
2026-05-18 21:28:59.728963: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:28:59.728989: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpi9trpzdm
2026-05-18 21:28:59.731312: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 21:28:59.770037: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpi9trpzdm
2026-05-18 21:28:59.781665: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 53785 m

In [29]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

pruned_loss, pruned_accuracy = stripped_pruned_model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Stripped pruned model test accuracy: {pruned_accuracy:.4f}")
print(f"Stripped pruned model test loss: {pruned_loss:.4f}")

y_pred_pruned = np.argmax(stripped_pruned_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification Report - Pruned Model")
print(classification_report(y_true, y_pred_pruned, target_names=wine.target_names))

print("Confusion Matrix - Pruned Model")
print(confusion_matrix(y_true, y_pred_pruned))

Stripped pruned model test accuracy: 0.9815
Stripped pruned model test loss: 0.3448

Classification Report - Pruned Model
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix - Pruned Model
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [30]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

tf.random.set_seed(42)
np.random.seed(42)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [31]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)

print("Teacher soft-label shape:", teacher_preds_soft.shape)
print("First teacher soft label:", teacher_preds_soft[0])

Teacher soft-label shape: (124, 3)
First teacher soft label: [9.9945003e-01 1.9636737e-04 3.5356887e-04]


In [32]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

alpha = 0.5
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

print("Combined distillation label shape:", y_train_combined.shape)

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1.0 - alpha) * soft_loss

Combined distillation label shape: (124, 6)


In [33]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_student = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 2s 44ms/step - loss: 1.1024 - accuracy: 0.3535 - val_loss: 1.0355 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 13ms/step - loss: 0.9323 - accuracy: 0.6364 - val_loss: 0.9208 - val_accuracy: 0.6800
Epoch 3/10
13/13 [==============================] - 0s 19ms/step - loss: 0.8162 - accuracy: 0.7980 - val_loss: 0.8263 - val_accuracy: 0.8000
Epoch 4/10
13/13 [==============================] - 0s 19ms/step - loss: 0.7113 - accuracy: 0.8687 - val_loss: 0.7181 - val_accuracy: 0.8800
Epoch 5/10
13/13 [==============================] - 0s 21ms/step - loss: 0.6081 - accuracy: 0.9091 - val_loss: 0.6061 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 19ms/step - loss: 0.5097 - accuracy: 0.9394 - val_loss: 0.5101 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 21ms/step - loss: 0.4229 - accuracy: 0.9495 - val_loss: 0.4303 - val_accuracy: 0.9200
Epoch 8/10
13

In [34]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model_kd = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_model_kd)

print(f"Knowledge-distilled student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmppomrz9he/assets


INFO:tensorflow:Assets written to: /tmp/tmppomrz9he/assets


Knowledge-distilled student TFLite model size: 6.10 KB


2026-05-18 21:34:04.012857: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-18 21:34:04.013037: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-18 21:34:04.013604: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmppomrz9he
2026-05-18 21:34:04.015879: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-18 21:34:04.015917: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmppomrz9he
2026-05-18 21:34:04.020137: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-18 21:34:04.094070: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmppomrz9he
2026-05-18 21:34:04.114478: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 100877 

In [35]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_student = np.argmax(student_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

student_accuracy = np.mean(y_pred_student == y_true)
print(f"Knowledge-distilled student test accuracy: {student_accuracy:.4f}")

print("\nClassification Report - Knowledge-Distilled Student")
print(classification_report(y_true, y_pred_student, target_names=wine.target_names))

print("Confusion Matrix - Knowledge-Distilled Student")
print(confusion_matrix(y_true, y_pred_student))

Knowledge-distilled student test accuracy: 0.9259

Classification Report - Knowledge-Distilled Student
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95        18
     class_1       0.95      0.86      0.90        21
     class_2       0.93      0.93      0.93        15

    accuracy                           0.93        54
   macro avg       0.93      0.93      0.93        54
weighted avg       0.93      0.93      0.93        54

Confusion Matrix - Knowledge-Distilled Student
[[18  0  0]
 [ 2 18  1]
 [ 0  1 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [36]:
# Part (e): Further size reduction experiment
tf.random.set_seed(42)
np.random.seed(42)

tiny_student_model = Sequential([
    Dense(16, activation='relu', input_shape=(num_features,)),
    Dense(8, activation='relu'),
    Dense(num_classes, activation='softmax')
])

tiny_teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
tiny_y_train_combined = np.concatenate([y_train_cat, tiny_teacher_preds_soft], axis=1)

def tiny_distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return 0.5 * hard_loss + 0.5 * soft_loss

tiny_student_model.compile(
    optimizer='adam',
    loss=tiny_distillation_loss,
    metrics=['accuracy']
)

history_tiny_student = tiny_student_model.fit(
    X_train_scaled,
    tiny_y_train_combined,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

tiny_student_eval_preds = np.argmax(tiny_student_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

tiny_student_accuracy = np.mean(tiny_student_eval_preds == y_true)
print(f"Tiny student Keras accuracy before quantization: {tiny_student_accuracy:.4f}")

# Full integer quantization of the tiny student.
converter = tf.lite.TFLiteConverter.from_keras_model(tiny_student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tiny_student_int8_tflite = converter.convert()

with open("model_tiny_student_int8.tflite", "wb") as f:
    f.write(tiny_student_int8_tflite)

print(f"Tiny student int8 TFLite model size: {file_size_kb('model_tiny_student_int8.tflite'):.2f} KB")

# Evaluate the tiny int8 model using the quantization evaluation helper.
tiny_quant_result = quantize_and_evaluate(
    tiny_student_model,
    X_test_scaled,
    y_test_cat,
    quant_type="int8",
    filename="model_tiny_student_int8.tflite"
)

Epoch 1/20
13/13 [==============================] - 12s 213ms/step - loss: 1.2217 - accuracy: 0.3333 - val_loss: 1.3579 - val_accuracy: 0.2000
Epoch 2/20
13/13 [==============================] - 1s 56ms/step - loss: 1.1204 - accuracy: 0.3535 - val_loss: 1.2570 - val_accuracy: 0.2000
Epoch 3/20
13/13 [==============================] - 1s 43ms/step - loss: 1.0371 - accuracy: 0.3939 - val_loss: 1.1798 - val_accuracy: 0.2000
Epoch 4/20
13/13 [==============================] - 1s 40ms/step - loss: 0.9653 - accuracy: 0.4545 - val_loss: 1.1152 - val_accuracy: 0.2000
Epoch 5/20
13/13 [==============================] - 1s 44ms/step - loss: 0.9030 - accuracy: 0.5152 - val_loss: 1.0500 - val_accuracy: 0.2800
Epoch 6/20
13/13 [==============================] - 1s 40ms/step - loss: 0.8465 - accuracy: 0.5758 - val_loss: 0.9909 - val_accuracy: 0.4000
Epoch 7/20
13/13 [==============================] - 0s 37ms/step - loss: 0.7968 - accuracy: 0.6263 - val_loss: 0.9334 - val_accuracy: 0.5200
Epoch 8/20


INFO:tensorflow:Assets written to: /tmp/tmpeo_x93qd/assets
/home/anany/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 10:23:17.266258: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 10:23:17.266725: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 10:23:17.274476: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpeo_x93qd
2026-05-20 10:23:17.278220: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 10:23:17.278316: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpeo_x93qd
2026-05-20 10:23:17.299062: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202

Tiny student int8 TFLite model size: 2.96 KB
INFO:tensorflow:Assets written to: /tmp/tmpcw2i3u7a/assets


INFO:tensorflow:Assets written to: /tmp/tmpcw2i3u7a/assets



INT8 TFLite model size: 2.96 KB
INT8 TFLite accuracy: 0.8148

Classification Report - INT8 TFLite
              precision    recall  f1-score   support

     class_0       0.69      1.00      0.82        18
     class_1       1.00      0.52      0.69        21
     class_2       0.88      1.00      0.94        15

    accuracy                           0.81        54
   macro avg       0.86      0.84      0.81        54
weighted avg       0.86      0.81      0.80        54

Confusion Matrix - INT8 TFLite
[[18  0  0]
 [ 8 11  2]
 [ 0  0 15]]


/home/anany/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 10:23:27.409219: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 10:23:27.409430: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 10:23:27.410232: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpcw2i3u7a
2026-05-20 10:23:27.416950: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 10:23:27.417092: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpcw2i3u7a
2026-05-20 10:23:27.432089: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 10:23:27.597918: I tensorflow/cc/saved_model/loader

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
